# 01 · The forward process — turning data into noise

Our plan is to destroy data with noise and learn to undo it. This notebook builds
the *destroying* half, carefully. By the end you'll have derived the single
equation that makes diffusion models practical.

> ### 📝 How this notebook works
> Cells marked **`# TODO`** are yours to write. Each is followed by a
> **self-check** cell that verifies your implementation and prints ✅.
>
> **Stuck?** The answer key is `solutions/notebooks/`, and the reference
> implementation lives in the `nanodiffusion/` package. Peeking is allowed —
> but try first.


## 1. One small step of noise

We corrupt the data over $T$ steps. A single step is defined as:

$$q(x_t \mid x_{t-1}) = \mathcal N\!\big(x_t;\ \underbrace{\sqrt{1-\beta_t}}_{\text{shrink}}\,x_{t-1},\ \underbrace{\beta_t I}_{\text{add noise}}\big)$$

In plain words, **two things happen at every step**:

1. **Shrink** the current sample slightly, multiplying by $\sqrt{1-\beta_t}$.
2. **Add** fresh Gaussian noise with variance $\beta_t$.

$\beta_t$ is a small number (like 0.0001 → 0.02) that we choose. The list
$\beta_1,\dots,\beta_T$ is called the **noise schedule**.

### Why shrink? (variance preservation)

The shrinking looks arbitrary, but it's doing something important. Recall that
for a constant $c$, $\operatorname{Var}(c\,x) = c^2\operatorname{Var}(x)$. So if
our data starts with variance 1:

$$\operatorname{Var}(x_t) = \underbrace{(1-\beta_t)\cdot 1}_{\text{from shrinking}} + \underbrace{\beta_t}_{\text{from new noise}} = 1$$

The variance stays exactly **1** forever. Without the shrink, we'd keep piling on
noise and the numbers would blow up. This design is called a
**variance-preserving** process — the data smoothly *morphs* into standard
Gaussian noise instead of exploding into it.

## 2. The reparameterization trick

Writing "$x_t$ is a *sample from* $\mathcal N(\mu, \sigma^2)$" is awkward for
algebra. The **reparameterization trick** rewrites any Gaussian sample as a
deterministic formula plus one standard noise draw:

$$x \sim \mathcal N(\mu, \sigma^2 I) \quad\Longleftrightarrow\quad x = \mu + \sigma\,\varepsilon,\qquad \varepsilon \sim \mathcal N(0, I)$$

All the randomness is isolated into $\varepsilon$. Applying it to our step:

$$\boxed{\;x_t = \sqrt{1-\beta_t}\;x_{t-1} + \sqrt{\beta_t}\;\varepsilon_t\;}$$

Now it's just algebra we can manipulate. (This trick is also what makes the whole
thing differentiable, which matters for training.)

**Notation.** From here on define $\alpha_t = 1-\beta_t$, so the step reads

$$x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon_t$$

## 3. Deriving the "nice property" ⭐

Here's the problem: to train, we need $x_t$ for a random $t$. Simulating 500
little steps every time would be painfully slow. Can we jump straight from $x_0$
to $x_t$?

Yes — and here's the derivation. We only need one fact from probability:

> **Sum of independent Gaussians:** if $a\sim\mathcal N(0,\sigma_a^2)$ and
> $b\sim\mathcal N(0,\sigma_b^2)$ are independent, then
> $a+b\sim\mathcal N(0,\ \sigma_a^2+\sigma_b^2)$. **Variances add.**

**Step 1 — write two consecutive steps:**

$$x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon_t$$
$$x_{t-1} = \sqrt{\alpha_{t-1}}\,x_{t-2} + \sqrt{1-\alpha_{t-1}}\,\varepsilon_{t-1}$$

**Step 2 — substitute the second into the first:**

$$x_t = \sqrt{\alpha_t}\Big(\sqrt{\alpha_{t-1}}\,x_{t-2} + \sqrt{1-\alpha_{t-1}}\,\varepsilon_{t-1}\Big) + \sqrt{1-\alpha_t}\,\varepsilon_t$$

$$x_t = \sqrt{\alpha_t\alpha_{t-1}}\;x_{t-2} \;+\; \underbrace{\sqrt{\alpha_t(1-\alpha_{t-1})}\,\varepsilon_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon_t}_{\text{two independent Gaussians}}$$

**Step 3 — merge the two noise terms.** Their variances add:

$$\alpha_t(1-\alpha_{t-1}) + (1-\alpha_t) = \alpha_t - \alpha_t\alpha_{t-1} + 1 - \alpha_t = 1 - \alpha_t\alpha_{t-1}$$

Look at that cancellation — the $\alpha_t$ terms vanish. So the two noises collapse
into a single one:

$$x_t = \sqrt{\alpha_t\alpha_{t-1}}\;x_{t-2} + \sqrt{1-\alpha_t\alpha_{t-1}}\;\varepsilon$$

**Step 4 — spot the pattern.** That has *exactly the same shape* as one step, but
with $\alpha_t\alpha_{t-1}$ in place of $\alpha_t$. Keep unrolling all the way to
$x_0$ and the products accumulate. Defining the running product

$$\bar\alpha_t = \prod_{s=1}^{t}\alpha_s$$

we get the **nice property**:

$$\boxed{\;x_t = \sqrt{\bar\alpha_t}\;x_0 \;+\; \sqrt{1-\bar\alpha_t}\;\varepsilon,\qquad \varepsilon\sim\mathcal N(0,I)\;}$$

**Why this is a big deal:** noising to *any* timestep is now **one line of code**,
$O(1)$ instead of $O(t)$. That's what lets us train on a random $t$ each step.

## 4. Reading $\bar\alpha_t$ as a signal dial

The nice property has a lovely interpretation. $x_t$ is a **weighted blend**:

$$x_t = \underbrace{\sqrt{\bar\alpha_t}}_{\text{how much signal}}\,x_0 + \underbrace{\sqrt{1-\bar\alpha_t}}_{\text{how much noise}}\,\varepsilon$$

and the two weights always satisfy $(\sqrt{\bar\alpha_t})^2+(\sqrt{1-\bar\alpha_t})^2=1$
— the variance-preservation from §1, now visible at every $t$.

| $\bar\alpha_t$ | meaning |
|---|---|
| $\approx 1$ (small $t$) | almost pure data, a whisper of noise |
| $\approx 0.5$ | half signal, half noise — the interesting middle |
| $\approx 0$ (large $t$) | signal erased; $x_T$ is indistinguishable from $\mathcal N(0,I)$ |

That last row is what lets us **start generation from pure noise**: if
$\bar\alpha_T\approx 0$, then noise is a valid starting point.

A useful summary number is the **signal-to-noise ratio**
$\mathrm{SNR}(t)=\dfrac{\bar\alpha_t}{1-\bar\alpha_t}$, which falls monotonically
from huge to ~0.

In [ ]:
import torch
import matplotlib.pyplot as plt

from nanodiffusion.utils import set_seed, scatter_2d
from nanodiffusion.data import toy2d
# reference implementations, used ONLY by the self-check cells:
from nanodiffusion.schedules import NoiseSchedule, linear_beta_schedule, cosine_beta_schedule
from nanodiffusion.forward import add_noise as reference_add_noise

set_seed(0)
T = 200
data = toy2d("swiss_roll", 8000)
print("data:", tuple(data.shape))

## TODO 1 — compute $\bar\alpha_t$ from the betas

Turn the schedule into the running product. Given `betas` of length $T$:

- $\alpha_t = 1-\beta_t$
- $\bar\alpha_t = \prod_{s\le t}\alpha_s$ — a **cumulative product**

*Hint:* `torch.cumprod(x, dim=0)` returns `[x0, x0*x1, x0*x1*x2, ...]`.

In [ ]:
def my_alpha_bars(betas: torch.Tensor) -> torch.Tensor:
    '''Return alpha_bar_t = prod_{s<=t} (1 - beta_s), same length as betas.'''
    # TODO: implement (2 lines)
    raise NotImplementedError

In [ ]:
# ---- self-check 1 ----
betas = linear_beta_schedule(T)
mine = my_alpha_bars(betas)
ref = NoiseSchedule(betas).alpha_bars
assert mine.shape == ref.shape, f"shape {mine.shape} != {ref.shape}"
assert torch.allclose(mine, ref, atol=1e-6), "values don't match the reference"
assert torch.all(mine[:-1] >= mine[1:]), "alpha_bar must be non-increasing"
print(f"✅ TODO 1 correct — signal decays from {mine[0]:.3f} to {mine[-1]:.4f}")
print("   (that's the *linear* schedule at T=200; §5 explains why it doesn't reach 0)")

## 5. Choosing the schedule: linear vs cosine

We get to pick the $\beta_t$. Two popular choices:

- **Linear** (original DDPM, 2020): $\beta_t$ ramps linearly from $10^{-4}$ to
  $0.02$ — a range tuned for $T=1000$ steps.
- **Cosine** (Improved DDPM, 2021): define $\bar\alpha_t$ *directly* from a cosine
  curve of the **fraction** $t/T$, then read the betas back out.

That difference — absolute $\beta$ range vs. a curve in $t/T$ — matters more than
it looks. Run the cell and compare the two columns.

**At $T=1000$** (what DDPM used) the linear schedule crushes the signal too
early: $\bar\alpha$ is already $0.08$ at the halfway point, and **327 of the 1000
steps** sit below $\bar\alpha<0.01$ — a third of the network's capacity spent on
inputs already indistinguishable from noise. Cosine holds $\bar\alpha\approx0.49$
at the midpoint and wastes only 65. *This* is the classic argument for cosine.

**At $T=200$** (what we use) linear has the **opposite** problem. Its $\beta$
range is hardcoded for 1000 steps, so across only 200 steps it never finishes the
job: $\bar\alpha_T = 0.13$, meaning $x_T$ still carries visible signal. That
quietly **breaks generation** — we start sampling from pure $\mathcal N(0,I)$, but
the forward process never actually got there, so the two ends don't meet.

Because cosine is defined in terms of $t/T$, it **adapts to whatever $T$ you
choose**, reaching $\bar\alpha_T\approx0$ in both columns. That's exactly why the
rest of these notebooks use **cosine with $T=200$** — it keeps cells fast without
breaking the math.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

# beta curves at our T
axes[0].plot(linear_beta_schedule(T), label="linear")
axes[0].plot(cosine_beta_schedule(T), label="cosine")
axes[0].set_title(r"$\beta_t$ (noise per step), $T=200$"); axes[0].set_xlabel("t"); axes[0].legend()

# alpha_bar at T=200 and T=1000, plotted against the FRACTION t/T so they compare
for ax, TT in zip(axes[1:], (200, 1000)):
    lin_ab = my_alpha_bars(linear_beta_schedule(TT))
    cos_ab = my_alpha_bars(cosine_beta_schedule(TT))
    frac = torch.linspace(0, 1, TT)
    ax.plot(frac, lin_ab, label=f"linear (ends {lin_ab[-1]:.3f})")
    ax.plot(frac, cos_ab, label=f"cosine (ends {cos_ab[-1]:.3f})")
    ax.axhline(0, ls=":", c="gray", lw=1)
    ax.set_title(r"$\bar\alpha_t$ (signal surviving), " + f"$T={TT}$")
    ax.set_xlabel("t / T"); ax.set_ylim(-0.05, 1.05); ax.legend()

plt.tight_layout(); plt.show()

for TT in (200, 1000):
    lin = my_alpha_bars(linear_beta_schedule(TT))
    cos = my_alpha_bars(cosine_beta_schedule(TT))
    print(f"T={TT:5d} | linear: mid={lin[TT//2]:.3f} end={lin[-1]:.4f} "
          f"wasted(<0.01)={int((lin < 0.01).sum()):4d}"
          f"   || cosine: mid={cos[TT//2]:.3f} end={cos[-1]:.4f} "
          f"wasted={int((cos < 0.01).sum()):4d}")

# our setting must actually reach pure noise, or sampling from N(0, I) is invalid
assert my_alpha_bars(cosine_beta_schedule(200))[-1] < 0.01
print("\ncosine @ T=200 reaches ~0 ✔  (linear @ T=200 would not)")

## TODO 2 — implement the nice property

Time to write the boxed equation from §3:

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$$

**One practical wrinkle: shapes.** Our data `x0` is `(B, 2)` and `t` is `(B,)` —
a *different* timestep per item. So `alpha_bars[t]` gives `(B,)`, one scalar per
item, and you must reshape it to `(B, 1)` before multiplying, so it broadcasts
across both coordinates of each point.

In [ ]:
def my_add_noise(x0: torch.Tensor, t: torch.Tensor, alpha_bars: torch.Tensor,
                 noise: torch.Tensor | None = None):
    '''Noise x0 to timestep t in one shot.

    Args:
        x0:         (B, 2) clean data
        t:          (B,) long tensor of timesteps
        alpha_bars: (T,) the cumulative products from TODO 1
        noise:      optional (B, 2) eps; sampled from N(0, I) if None
    Returns:
        (x_t, noise)   -- we return the noise too, because training needs
                          the exact eps the network will be asked to predict.
    '''
    if noise is None:
        noise = torch.randn_like(x0)
    # TODO:
    #   1. gather alpha_bar_t for each item:      ab = alpha_bars[t]
    #   2. reshape it to (B, 1) so it broadcasts: ab = ab.reshape(-1, 1)
    #   3. return  sqrt(ab) * x0 + sqrt(1 - ab) * noise,  and the noise
    raise NotImplementedError

In [ ]:
# ---- self-check 2 ----
schedule = NoiseSchedule.make("cosine", T)
ab = my_alpha_bars(cosine_beta_schedule(T))
t = torch.randint(0, T, (data.shape[0],))
eps = torch.randn_like(data)                      # fixed noise, so we can compare

x_mine, _ = my_add_noise(data, t, ab, noise=eps)
x_ref, _ = reference_add_noise(data, t, schedule, noise=eps)
assert x_mine.shape == data.shape, f"shape {x_mine.shape} != {data.shape}"
assert torch.allclose(x_mine, x_ref, atol=1e-5), "doesn't match the reference"

# at t = T-1 the data should be ~pure unit-variance noise (this is what makes it
# legitimate to *start* generation from N(0, I) later)
x_T, _ = my_add_noise(data, torch.full((data.shape[0],), T - 1), ab)
print(f"alpha_bar_T = {ab[-1]:.4f}  (want ~0)")
print(f"std(x_T)    = {x_T.std().item():.4f}  (want ~1)")
assert abs(x_T.std().item() - 1.0) < 0.15
print("✅ TODO 2 correct — the forward process works")

## 6. See it work

Watch the swiss roll dissolve using **your** implementation. Notice it doesn't
drift or explode — it *morphs* into a unit Gaussian blob. That's variance
preservation doing its job.

In [ ]:
ts = [0, 10, 25, 50, 100, 199]
fig, axes = plt.subplots(1, len(ts), figsize=(2.6 * len(ts), 2.6))
for ax, ti in zip(axes, ts):
    x_t, _ = my_add_noise(data, torch.full((data.shape[0],), ti), ab)
    scatter_2d(ax, x_t, f"t = {ti}   " + r"$\bar\alpha$=" + f"{ab[ti]:.2f}")
plt.suptitle("Your forward process: swiss roll -> noise")
plt.tight_layout(); plt.show()

## Recap

- A step **shrinks then adds noise**, keeping variance at 1 (variance preserving).
- The **reparameterization trick** turns sampling into algebra.
- Because **variances of independent Gaussians add**, $t$ steps collapse into one:
  $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$ — the **nice property**.
- $\bar\alpha_t$ is the signal dial; at $t=T$ it's ~0, so $x_T$ is just noise.

**The open question for notebook 02:** given a noisy $x_t$, how do we step
*backwards*? (Spoiler: it comes down to guessing the $\varepsilon$ in that boxed
equation.)